In [ ]:
import torch
import torch.nn.functional as F
from importlib import reload
import datasets
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import json, os
from itertools import combinations
import analysis_helpers
from get_routing_weights import DATA_FOLDER

In [ ]:
output_filepath = os.path.join(DATA_FOLDER, "expert_importance_flores_qwen35_15langs.json")
with open(output_filepath, 'r') as f:
    reloaded_data = json.load(f)

reloaded_tensor_data = {}
for key, list_of_arrays in reloaded_data.items():
    tensor_list = []
    for array in list_of_arrays:
        tensor_list.append(torch.tensor(array))
    reloaded_tensor_data[key] = tensor_list

In [ ]:
def analyze_routing_entropy(expert_importance):
    """
    Analyze routing entropy across languages and layers.
    
    Args:
        expert_importance: dict where keys are languages and values are lists of 
                             Y x E tensors (Y layers, E experts)
    
    Returns:
        results: dict where keys are languages and values are mean entropy per layer (Y,)
    """
    results = {}
    
    for language, tensor_list in expert_importance.items():
        print(f"Processing {language}...")
        
        # List to store entropy for each string
        entropy_per_string = []
        
        for tensor_idx, routing_tensor in enumerate(tensor_list):
            # routing_tensor shape: (Y layers, E experts)
            # Calculate entropy for each layer
            layer_entropies = analysis_helpers.calculate_entropy(routing_tensor)  # Shape: (Y,)
            entropy_per_string.append(layer_entropies)
        
        # Stack all entropies and calculate mean across strings
        # Shape: (N strings, Y layers) -> mean over N -> (Y layers,)
        all_entropies = torch.stack(entropy_per_string, dim=0)
        mean_entropy_per_layer = torch.mean(all_entropies, dim=0)
        
        results[language] = mean_entropy_per_layer
        print(f"  {language}: {len(tensor_list)} strings, {len(mean_entropy_per_layer)} layers")
    
    return results

entropy_results = analyze_routing_entropy(reloaded_tensor_data)

In [ ]:
mean_jsds_per_layer_from_eng['pes']

In [ ]:
mean_jsds_per_layer_from_eng = analysis_helpers.pairwise_mean_jsd(reloaded_tensor_data, entropy_normalized=True)

In [ ]:
arb_tensor = mean_jsds_per_layer_from_eng.pop('arb')

In [ ]:
fr_code = 'fr' if len(mean_jsds_per_layer_from_eng.keys())==9 else 'fra'
N = int(mean_jsds_per_layer_from_eng[fr_code].size()[0])

# --- Prepare data for plotting ---
# The x-axis will represent the layer index. We can make it 1-indexed for better readability.
layer_indices = np.arange(1, N + 1) # Creates an array from 1 to 48

# Convert tensor to a NumPy array if it's on GPU or if matplotlib prefers it

# --- Create the line graph ---
plt.figure(figsize=(12, 6)) # Adjust figure size for better readability of 48 points
for lang in mean_jsds_per_layer_from_eng.keys():
    jsd_values = mean_jsds_per_layer_from_eng[lang].to(torch.float32).cpu().numpy()
    plt.plot(layer_indices, jsd_values, marker='o', linestyle='-', linewidth=2, markersize=5, label=lang)

# --- Add labels, title, and grid for clarity ---
plt.suptitle('Mean JSD (entropy-normalized) per Layer', fontsize=16)
plt.title('Comparison to English Routing weights (Qwen3.5 35B A3B)')
plt.legend()
plt.xlabel('Layer Index', fontsize=12)
plt.ylabel('Mean JSD', fontsize=12)
plt.xticks(np.arange(1, N+1, 5)) # Set x-axis ticks to show every 5 layers for readability
plt.grid(True, linestyle='--', alpha=0.7)
plt.ylim(bottom=0, top=1.0) # JSD is always non-negative, so set y-axis lower limit to 0


# --- Display the plot ---
plt.tight_layout() # Adjusts plot to prevent labels from overlapping
plt.show()

In [ ]:
# Get a color map with 15 distinct colors.
# 'tab20' is a good choice as it has 20 unique colors.
colors = dict(zip(mean_jsds_per_layer_from_eng.keys(), plt.cm.get_cmap('tab20')(np.linspace(0, 1, len(mean_jsds_per_layer_from_eng.keys())))))

scale_factor = 2
plt.figure(figsize=(9 * scale_factor, 4 * scale_factor)) # Adjust figure size for better readability of 48 points
for i, lang in enumerate(mean_jsds_per_layer_from_eng.keys()):
    jsd_values = mean_jsds_per_layer_from_eng[lang].to(torch.float32).cpu().numpy()
    plt.plot(layer_indices, jsd_values, marker='o', linestyle='-', linewidth=2  * scale_factor, markersize=5  * scale_factor, label=lang, color=colors[lang])

# --- Add labels, title, and grid for clarity ---
plt.title(
    "[ Qwen3.5 35B A3B ]",
    y=0.85, fontsize=14 * scale_factor,
    bbox=dict(facecolor='white', edgecolor='none', boxstyle='square,pad=0.4', alpha=1.0)
)
plt.suptitle(
    'Routing Divergence from English, per Layer', 
    y=0.9, fontsize=14 * scale_factor, fontweight='bold', 
    bbox=dict(facecolor='white', edgecolor='none', boxstyle='square,pad=0.2', alpha=1.0)
)
plt.legend(bbox_to_anchor=(1, 1), loc='upper left', borderaxespad=0., fontsize=9 * scale_factor)
plt.xlabel('Layer Number', fontsize=12 * scale_factor)
plt.ylabel('Mean JSD (entropy-normalized)', fontsize=12 * scale_factor)
xticks = np.append(np.arange(1, N+1, 5), N)
plt.xticks(xticks) # Set x-axis ticks to show every 5 layers for readability
plt.grid(True, linestyle='--', alpha=0.7)
plt.yticks(np.arange(0, 0.6, 0.05))
plt.xlim(left=0.5, right=N+0.5)
plt.ylim(bottom=0, top=0.55) # JSD is always non-negative, so set y-axis lower limit to 0


# Increase the tick size for x and y axes
plt.tick_params(axis='both', which='major', labelsize=9 * scale_factor)
ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)

# --- Display the plot ---
plt.tight_layout() # Adjusts plot to prevent labels from overlapping
plt.show()

In [ ]:
def plot_entropy_by_language(entropy_results, model_pretty_name):
    """
    Plot mean entropy per layer for each language.
    
    Args:
        entropy_results: dict from analyze_routing_entropy()
    """
    plt.figure(figsize=(16, 12))
    
    # Get number of layers (assuming all languages have same number)
    num_layers = len(next(iter(entropy_results.values())))
    layer_numbers = range(1, num_layers + 1)
    
    # Plot each language
    for language, mean_entropies in entropy_results.items():
        # Convert to numpy for plotting
        entropies_np = mean_entropies.detach().cpu().numpy()
        if language in colors.keys():
            plt.plot(layer_numbers, entropies_np, marker='o', label=language, linewidth=4, markersize=10, color=colors[language])
        else:
            c = 'black' if language == 'eng' else 'purple'
            plt.plot(layer_numbers, entropies_np, marker='o', label=language, linewidth=4, markersize=10, color=c)
    
    plt.xlabel('Layer Number', fontsize=24)
    plt.ylabel('Mean Routing Entropy', fontsize=24)
    plt.title(f'Mean Routing Entropy, per Layer [ {model_pretty_name} ]', fontsize=36, fontweight='bold')
    plt.legend(bbox_to_anchor=(1, 1), loc='upper left', fontsize=18)
    xticks = np.append(np.arange(1, N+1, 5), N)
    plt.xticks(xticks) # Set x-axis ticks to show every 5 layers for readability
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.xlim(left=0.5, right=N+0.5)
    plt.tight_layout()
    
    # Add some styling
    plt.tick_params(axis='both', which='major', labelsize=18)
    ax = plt.gca()
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)

    for m4 in range(4, 41, 4):
        ax.axvline(x=m4, color='red', linestyle='--', linewidth=1, alpha=0.5)
    
    return plt.gcf()

fig = plot_entropy_by_language(entropy_results, model_pretty_name='Qwen3.5 35B A3B')
plt.show()

In [ ]:
from matplotlib.colors import LinearSegmentedColormap, Normalize

cmap_colors = [(0.0, 'red'), (0.74, 'yellow'), (1.0, 'green')]
custom_cmap = LinearSegmentedColormap.from_list('custom_RdYlGn', cmap_colors)

# Then, in your code, you would use this cmap with your existing Normalize object
norm = Normalize(vmin=0.3, vmax=0.9)
sm = plt.cm.ScalarMappable(cmap=custom_cmap, norm=norm)
sm.set_array([])

metric_data = {
    "belebele_arb_Arab": 0.882,
    "belebele_arb_Latn": 0.322,
    "belebele_ary_Arab": 0.588,
    "belebele_asm_Beng": 0.701,
    "belebele_bam_Latn": 0.306,
    "belebele_ben_Beng": 0.799,
    "belebele_eng_Latn": 0.918,
    "belebele_fra_Latn": 0.903,
    "belebele_hin_Deva": 0.757,
    "belebele_lit_Latn": 0.858,
    "belebele_ory_Orya": 0.732,
    "belebele_pes_Arab": 0.822,
    "belebele_srp_Cyrl": 0.865,
    "belebele_tha_Thai": 0.832,
    "belebele_zho_Hans": 0.898
}



scale_factor = 2
plt.figure(figsize=(9 * scale_factor, 4 * scale_factor)) # Adjust figure size for better readability of 48 points
# for i, lang in enumerate(mean_jsds_per_layer_from_eng.keys()):
for lang, metric_value in metric_data.items():
    if lang[9:12] in mean_jsds_per_layer_from_eng.keys():
        lang = lang[9:12]
    else:
        continue
    jsd_values = mean_jsds_per_layer_from_eng[lang].to(torch.float32).cpu().numpy()
    line_color = sm.to_rgba(metric_value)
    plt.plot(layer_indices, jsd_values, marker='o', linestyle='-', linewidth=2  * scale_factor, markersize=5  * scale_factor, label=lang, color=line_color)

# --- Add labels, title, and grid for clarity ---
# plt.title('Routing Divergence from English, per Layer [ Qwen3-30B-A3B ]', fontsize=16 * scale_factor)
# plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)
plt.xlabel('Layer Number', fontsize=12 * scale_factor)
plt.ylabel('Mean JSD (entropy-normalized)', fontsize=12 * scale_factor)
xticks = np.append(np.arange(1, N+1, 7), 48)
plt.xticks(xticks) # Set x-axis ticks to show every 5 layers for readability
plt.grid(True, linestyle='--', alpha=0.7)
plt.yticks(np.arange(0, 0.45, 0.05))
plt.xlim(left=0.5, right=48.5)
plt.ylim(bottom=0, top=0.5) # JSD is always non-negative, so set y-axis lower limit to 0


# Increase the tick size for x and y axes
plt.tick_params(axis='both', which='major', labelsize=8 * scale_factor)
ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)

# --- Add the color bar for the metric ---
cbar = fig.colorbar(sm, ax=ax, orientation='vertical', pad=0.01)
cbar.ax.tick_params(labelsize=7 * scale_factor)
cbar.set_label('Belebele 0-shot accuracy', fontsize=12 * scale_factor)

# --- Display the plot ---
plt.tight_layout() # Adjusts plot to prevent labels from overlapping
plt.show()

In [ ]:
mean_jsds_per_layer_pairwise = analysis_helpers.pairwise_mean_jsd(reloaded_tensor_data, False, True)

In [ ]:
all_unique_languages = ['tha', 'zho', 'arb', 'ary', 'pes', 'asm', 'ben', 'hin', 'ory', 'bam', 'lit', 'srp', 'fra', 'eng']
lang_to_idx = {lang: i for i, lang in enumerate(all_unique_languages)}
num_unique_languages = len(all_unique_languages)

print(f"\nUnique languages found: {all_unique_languages}")
print(f"Number of unique languages: {num_unique_languages}")

# --- 3. Create JSD matrices for layer 8 and layer 16 ---
# Layers are 0-indexed in Python, so layer 8 is index 7, and layer 16 is index 15.
target_layers = [16, 28]
jsd_matrices = {layer: np.zeros((num_unique_languages, num_unique_languages)) for layer in target_layers}

# Populate the matrices
for (lang1, lang2), jsd_values_per_layer in mean_jsds_per_layer_pairwise.items():
    idx1, idx2 = lang_to_idx[lang1], lang_to_idx[lang2]

    for layer_idx in target_layers:
        if 0 <= layer_idx-1 < len(jsd_values_per_layer):
            jsd_val = jsd_values_per_layer[layer_idx-1]
            jsd_matrices[layer_idx][idx1, idx2] = jsd_val
            jsd_matrices[layer_idx][idx2, idx1] = jsd_val # JSD is symmetric

# Set diagonal to 0 (JSD of a distribution with itself is 0)
for layer_idx in target_layers:
    np.fill_diagonal(jsd_matrices[layer_idx], 0)
mask_diagonal = np.identity(num_unique_languages, dtype=bool)

# --- 4. Plot two heatmaps ---
fig, axes = plt.subplots(1, 2, figsize=(18, 8)) # 1 row, 2 columns for subplots

# Define a common color map and value range for consistency
vmax_val = max(matrix.max() for matrix in jsd_matrices.values()) # Max JSD across both matrices
vmin_val = 0 # JSD is always non-negative
cmap_name = "viridis" # Good default, or try "magma", "coolwarm"

for i, layer_idx in enumerate(target_layers):
    ax = axes[i]
    sns.heatmap(
        jsd_matrices[layer_idx],
        annot=True,        # Show the JSD values on the heatmap
        fmt=".2f",         # Format annotations to 2 decimal places
        cmap=cmap_name,    # Color map
        linewidths=.5,     # Add lines between cells
        linecolor='lightgray',
        cbar=True,         # Show color bar
        vmin=vmin_val,     # Set common minimum for color bar
        vmax=vmax_val,     # Set common maximum for color bar
        ax=ax,
        mask=mask_diagonal
    )
    ax.set_title(f'Pairwise JSD for Layer {layer_idx}', fontsize=16) # +1 to make it 1-indexed for display
    ax.set_xlabel('Language', fontsize=12)
    ax.set_ylabel('Language', fontsize=12)
    ax.set_xticklabels(all_unique_languages, rotation=90) # Rotate labels if many languages
    ax.set_yticklabels(all_unique_languages, rotation=0)
plt.suptitle("Inter-language routing similarity [Qwen3-30B-A3B]")
plt.tight_layout()
plt.show()

In [ ]:
reloaded_tensor_data.keys()

# Identifying multilingual-specialized experts

### Using Expert Importance

In [ ]:
TARGET_LAYERS = range(8, 35) # one-indexed, middle layers

aggressive_deactivation = dict()
for i in TARGET_LAYERS:
    layer_idx = i-1 # now zero-indexed
    # router_logits_per_lang_per_token (488, 48, 128, S)
    # reloaded_tensor_data['eng'] (488, 48, 128)
    # softmax -> (488, 48, 128) (probabilities)
    layer_avg_prob_eng = F.softmax(torch.stack(reloaded_tensor_data['eng']), dim=-1).mean(dim=0)[layer_idx]
    # layer_avg_prob_eng (48, 128)

    identified_experts = set()

    # FIRST find experts where for any language, has 3 SDs above 0 difference avg prob from english
    probs_by_lang = dict()
    for lang, data in reloaded_tensor_data.items():
        if 'eng' == lang:
            continue
        layer_avg_prob_lang = F.softmax(torch.stack(data), dim=-1).mean(dim=0)[layer_idx]
        diff = layer_avg_prob_lang - layer_avg_prob_eng
        experts = torch.where(diff > 3 * diff.std())[0]
        for exp in experts:
            identified_experts.add(int(exp))
        probs_by_lang[lang] = layer_avg_prob_lang
        
    
    # SECOND find experts where for an aggregated average of all languages, has 3 SDs above 0 difference avg prob from english
    multiling_weight = {
        'bam': 0.5, # too low-resource
        'fra': 0.5, # too high-resource
        'tha': 1.0, 
        'pes': 1.0, 
        'arb': 1.0, 
        'ary': 0.5, # too low-resource 
        'hin': 1.0, 
        'zho': 0.5, # too high-resource 
        'srp': 1.0, 
        'lit': 1.0, 
        'ory': 0.5, # too low-resource 
        'ben': 1.0, 
        'asm': 0.5, # too low-resource
    }

    langs = list(probs_by_lang.keys())
    tensor_stack = torch.stack([probs_by_lang[k] for k in langs])
    weight_tensor = torch.tensor([multiling_weight[k] for k in langs])
    weight_tensor = weight_tensor / weight_tensor.sum()
    multilingual_weighted_sum = torch.sum(tensor_stack * weight_tensor.unsqueeze(-1), dim=0)
    diff = multilingual_weighted_sum - layer_avg_prob_eng
    experts = torch.where(diff > 3 * diff.std())[0]
    for exp in experts:
        if int(exp) not in identified_experts:
            print("not prev identified", layer_idx, exp)
        identified_experts.add(int(exp))
    aggressive_deactivation[str(layer_idx)] = list(identified_experts)
    


In [ ]:
with open("expert_steering/aggressive_deactivation.json", 'w') as f:
    json.dump(aggressive_deactivation, f, indent=4)

In [ ]:
l7_avg_prob_ben = F.softmax(torch.stack(reloaded_tensor_data['fra']), dim=-1).mean(dim=0)[7]
l7_avg_prob_eng = F.softmax(torch.stack(reloaded_tensor_data['eng']), dim=-1).mean(dim=0)[7]
eng_topk = l7_avg_prob_eng.topk(8 * 2)

diff = (l7_avg_prob_ben - l7_avg_prob_eng)


In [ ]:
from expert_steering.expert_identification import load_activation_counts_data, calculate_outlier_coefficients_from_expert_importance

In [ ]:
coefficients = calculate_outlier_coefficients_from_expert_importance('mgsminstruct', target_lang='bn')

In [ ]:
diff = coefficients[14]
coefficients.shape

In [ ]:
# Create colors based on sign and magnitude
colors = ['red' if x < 0 else 'green' for x in diff]

# Create bar plot
plt.figure(figsize=(10, 6))
bars = plt.bar(range(len(diff)), diff, color=colors, alpha=0.7)

# Optional: Make color intensity proportional to magnitude
max_abs = np.abs(diff).max()
for bar, val in zip(bars, diff):
    intensity = abs(val) / max_abs
    # if val < 0:
    #     bar.set_color((1, 0, 0, 0.3 + 0.7 * intensity))  # Red with varying alpha
    # else:
    #     bar.set_color((0, 0.8, 0, 0.3 + 0.7 * intensity))  # Green with varying alpha

plt.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
plt.xlabel('Index')
plt.ylabel('Value')
plt.title('Difference in Avg Expert Importance for Bengali vs English (Qwen3 30B, layer 14)')
plt.grid(True, alpha=0.3)
plt.show()

### Using Activation Counts

In [ ]:
output_filepath = "collected_routing_data/activation_counts_mgsminstruct_qwen3_30b_11langs.json"
with open(output_filepath, 'r') as f:
    reloaded_data = json.load(f)

reloaded_tensor_data = {}
for key, list_of_arrays in reloaded_data.items():
    tensor_list = []
    for array in list_of_arrays:
        tensor_list.append(torch.tensor(array))
    reloaded_tensor_data[key] = tensor_list

In [ ]:
# calculate relative frequency of expert activation (pct of inputs where expert is activated)
stacked_tensor = torch.stack(reloaded_tensor_data['en']) # Shape: (num_sequences, num_layers, num_experts)
activation_relative_frequency = stacked_tensor / stacked_tensor.sum(dim=-1, keepdim=True) * 8 # Shape: (num_sequences, num_layers, num_experts)
avg_arf_eng = activation_relative_frequency.mean(dim=0) # Shape: (num_layers, num_experts)

tgt = 'bn'
stacked_tensor = torch.stack(reloaded_tensor_data[tgt]) # Shape: (num_sequences, num_layers, num_experts)
activation_frequency_relative = stacked_tensor / stacked_tensor.sum(dim=-1, keepdim=True) * 8 # Shape: (num_sequences, num_layers, num_experts)
avg_arf_tgt = activation_frequency_relative.mean(dim=0) # Shape: (num_layers, num_experts)

diff = avg_arf_tgt - avg_arf_eng

The `outlier_coefficients` is a tensor of shape [48,128]


In [ ]:
torch.save(outlier_coefficients, 'expert_outlier_coefficient.pt')

In [ ]:
plt.plot(diff.std(dim=1))

In [ ]:
stds = diff.std(dim=1, keepdim=True)
normed_diff = diff / stds
thresh1 = 0.3
thresh2 = 3
counts = (diff > thresh1).sum(dim=1)
normed_counts = (normed_diff > thresh2).sum(dim=1)

plt.plot(counts, label='diff experts')
plt.plot(normed_counts, label='normed diff experts')
plt.title("Num Experts per layer using Raw Diff vs SD-normed Diff")
plt.legend()
# # Convert to numpy for plotting
# x = np.arange(len(stds))
# y1 = normed_counts.numpy()
# y2 = counts.numpy()

# # Create the plot with dual y-axes
# fig, ax1 = plt.subplots(figsize=(10, 6))

# # Plot first tensor on left y-axis
# color1 = 'tab:blue'
# ax1.set_xlabel('Index')
# ax1.set_ylabel(f'Num Experts w/ normed diff > {thresh2}', color=color1)
# line1 = ax1.plot(x, y1, color=color1, label='Num Experts')
# ax1.tick_params(axis='y', labelcolor=color1)

# # Create second y-axis and plot second tensor
# ax2 = ax1.twinx()
# color2 = 'tab:red'
# ax2.set_ylabel(f'Num Experts w/ diff > {thresh1}', color=color2)
# line2 = ax2.plot(x, y2, color=color2, label='Num Experts')
# ax2.tick_params(axis='y', labelcolor=color2)

# # Add legend
# lines1, labels1 = ax1.get_legend_handles_labels()
# lines2, labels2 = ax2.get_legend_handles_labels()
# ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

# plt.title('Overlay of Two Tensors with Different Scales')
# plt.tight_layout()
# plt.show()

In [ ]:
thresh = 0.2
counts = (diff > 0.3).sum(dim=1)
counts

In [ ]:
def determine_deactivated_experts(coefficients_tensor, thresh, layers_ommitted):
    deactivation_dict = {}

    num_layers = coefficients_tensor.shape[0]
    for layer_idx in range(coefficients_tensor.shape[0]):
        if layer_idx < layers_ommitted or layer_idx >= (num_layers - layers_ommitted):
            continue # edge layers ommitted

        # Get experts where values > thresh for this layer
        expert_idxs = torch.where(coefficients_tensor[layer_idx] > thresh)[0].tolist()
        deactivation_dict[str(layer_idx)] = expert_idxs
    return deactivation_dict

In [ ]:
layer_idx = 47
stacked_tensor = torch.stack(reloaded_tensor_data['en'])
activation_frequency_relative = stacked_tensor / stacked_tensor.sum(dim=-1, keepdim=True) * 8
layer_avg_prob_eng = activation_frequency_relative.mean(dim=0)[layer_idx]
stacked_tensor = torch.stack(reloaded_tensor_data['bn'])
activation_frequency_relative = stacked_tensor / stacked_tensor.sum(dim=-1, keepdim=True) * 8
layer_avg_prob_ben = activation_frequency_relative.mean(dim=0)[layer_idx]
diff = layer_avg_prob_ben - layer_avg_prob_eng


# Create colors based on sign and magnitude
colors = ['red' if x < 0 else 'green' for x in diff]

# Create bar plot
plt.figure(figsize=(10, 6))
bars = plt.bar(range(len(diff)), diff, color=colors, alpha=0.7)

# Optional: Make color intensity proportional to magnitude
max_abs = np.abs(diff).max()
for bar, val in zip(bars, diff):
    intensity = abs(val) / max_abs
    # if val < 0:
    #     bar.set_color((1, 0, 0, 0.3 + 0.7 * intensity))  # Red with varying alpha
    # else:
    #     bar.set_color((0, 0.8, 0, 0.3 + 0.7 * intensity))  # Green with varying alpha

plt.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
plt.xlabel('Experts')
plt.ylabel('Difference in relative frequency')
plt.suptitle('Difference in Qwen3 Expert Activation (Eng-Ben) on Parallel Math data', fontsize=16)
plt.title(f'Green/positive values are experts more often activated in Bengali than English (layer:{layer_idx+1})')
plt.grid(True, alpha=0.3)
plt.show()